In [3]:
import pandas as pd

# Lien vers le fichier DVF compressé
url = "https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres-geolocalisees/20260717-142041/dvf.csv.gz"

premier = True

# Lecture progressive du fichier
for dvf_dept in pd.read_csv(url, sep=",", compression="gzip", chunksize=100_000, low_memory=False, dtype={"code_departement": "string"}):
    # Garder uniquement les lignes du département 69
    dept_69 = dvf_dept[dvf_dept["code_departement"] == "69"]

    # Écrire uniquement les chunks non vides
    if not dept_69.empty:
        dept_69.to_csv( "../data/raw/dvf_69.csv" ,mode="w" if premier else "a", header=premier, index=False )
        premier = False

print("Extraction du département 69 terminée.")

Extraction du département 69 terminée.


In [ ]:
import pandas as pd

# Charger les données INSEE
df_insee = pd.read_csv("../data/raw/comparateur_territoires.csv", sep=";")

# Mettre les codes géographiques au format texte sur 5 caractères
df_insee["GEO"] = df_insee["GEO"].astype(str).str.zfill(5)

# Définir les codes des 9 arrondissements de Lyon
codes_lyon = ["69381", "69382", "69383", "69384", "69385", "69386", "69387", "69388", "69389"]

# Garder les communes du 69 et les 9 arrondissements de Lyon (car on ce detail dans DVF )
df_rhone = df_insee[((df_insee["GEO_OBJECT"] == "COM") & (df_insee["GEO"].str.startswith("69"))) | 
            ((df_insee["GEO_OBJECT"] == "ARM") & (df_insee["GEO"].isin(codes_lyon)))
].copy()

print("Dimensions :", df_rhone.shape)
print("Nombre de territoires :", df_rhone["GEO"].nunique())
print("nombre de communes et d'arrondissements: ", df_rhone.groupby("GEO_OBJECT")["GEO"].nunique())

# Afficher les 9 codes des arrondissements de Lyon
print(df_rhone.loc[df_rhone["GEO_OBJECT"] == "ARM", "GEO"].drop_duplicates().sort_values().tolist())

display(df_rhone.head())

# Sauvegarde
df_rhone.to_csv("../data/raw/insee_comparateur_territoires_69.csv", index=False)

print("Fichier comparateur_territoires_rhone.csv enregistré.")


/tmp/ipykernel_7119/174876249.py:4: DtypeWarning: Columns (0: GEO) have mixed types. Specify dtype option on import or set low_memory=False.
  df_insee = pd.read_csv("../data/raw/comparateur_territoires.csv", sep=";")


Dimensions : (19250, 11)
Nombre de territoires : 275
nombre de communes et d'arrondissements:  GEO_OBJECT
ARM      9
COM    266
Name: GEO, dtype: int64
['69381', '69382', '69383', '69384', '69385', '69386', '69387', '69388', '69389']


,GEO_REF,GEO_OBJECT,GEO,TIME_PERIOD,DS,TAB_MEASURE,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,UNIT_MULT,CONF_STATUS
50016,2026,ARM,69381,2025,DS_ETAT_CIVIL_NAIS_COMMUNES,LVB,242.0,A,NaN,NaN,NaN
50017,2026,ARM,69381,2025,DS_ETAT_CIVIL_DECES_COMMUNES,DTH,169.0,A,NaN,NaN,NaN
50018,2026,ARM,69381,2024,DS_ETAT_CIVIL_NAIS_COMMUNES,LVB,255.0,A,NaN,NaN,NaN
50019,2026,ARM,69381,2024,DS_ETAT_CIVIL_DECES_COMMUNES,DTH,184.0,A,NaN,NaN,NaN
50020,2026,ARM,69381,2024,DS_FLORES_A5,UNIT_LOC_NUMBER_EMPL_E100T199,8.0,A,NaN,NaN,NaN


Fichier comparateur_territoires_rhone.csv enregistré.


In [ ]:
import pandas as pd

referentiel = pd.read_csv("../data/processed/referentiel_communes_rhone.csv", dtype={"code_commune": "string"})
dvf = pd.read_csv("../data/processed/dvf_nettoye.csv", dtype={"code_commune": "string"})

referentiel["code_commune"] = referentiel["code_commune"].str.zfill(5)
dvf["code_commune"] = dvf["code_commune"].str.zfill(5)

coordonnees = dvf.groupby("code_commune", as_index=False).agg(latitude=("latitude", "median"), longitude=("longitude", "median"))

referentiel = referentiel.merge(coordonnees, on="code_commune", how="left")

referentiel.to_csv("../data/processed/referentiel.csv", index=False)

display(referentiel.head())